# 05. Transformer Representations for LLM Attribution

Compare **frozen RoBERTa embeddings** with character TF-IDF and explicit stylometry for 18-model attribution. Evaluation uses the same five prompt-grouped folds as notebooks 01–04.

## 1. Setup and data

In [1]:
from pathlib import Path
import hashlib
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display

import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.svm import LinearSVC

try:
    from transformers import AutoModel, AutoTokenizer
except ImportError as exc:
    raise ImportError("Install transformers: pip install transformers") from exc

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

candidates = [
    ROOT / "data" / "train.csv",
    ROOT / "train.csv",
    ROOT / "train(1).csv",
    Path("/mnt/data/train(1).csv"),
]
TRAIN_PATH = next((p for p in candidates if p.exists()), None)
if TRAIN_PATH is None:
    raise FileNotFoundError("Put train.csv in data/ or beside this notebook")

df = pd.read_csv(TRAIN_PATH).reset_index(drop=True)
assert {"text", "model", "prompt_id"}.issubset(df.columns)
X = df["text"].fillna("").astype(str)
y = df["model"].astype(str)
groups = df["prompt_id"]
LABELS = sorted(y.unique())
folds = list(GroupKFold(n_splits=5).split(X, y, groups))
assert all(set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te])) for tr, te in folds)

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print(f"{len(df)} texts, {len(LABELS)} labels, {groups.nunique()} prompts")
print("Device:", device)

/Users/ulad/kaggle/name-that-llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2699 texts, 18 labels, 150 prompts
Device: mps


## 2. Frozen RoBERTa embeddings

Use **mean pooling of the final hidden states** over non-padding tokens. RoBERTa weights remain frozen. Embeddings can be computed once for all texts because the encoder is pretrained and receives no dataset labels. For the local Mac, start with `BATCH_SIZE=16`; increase if memory permits.

In [2]:
MODEL_ID = "FacebookAI/roberta-base"   # or path to an attached local checkpoint
MAX_LENGTH = 256
BATCH_SIZE = 16
CACHE_DIR = ROOT / ".cache" / "llm_attribution"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

fingerprint = hashlib.sha256(
    (MODEL_ID + "|" + str(MAX_LENGTH) + "|" + "\n".join(X.tolist())).encode("utf-8")
).hexdigest()[:16]
EMB_PATH = CACHE_DIR / f"roberta_mean_{fingerprint}.npy"

if EMB_PATH.exists():
    embeddings = np.load(EMB_PATH)
    print("Loaded cached embeddings:", EMB_PATH)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    encoder = AutoModel.from_pretrained(MODEL_ID).to(device)
    encoder.eval()
    arrays = []
    from tqdm.auto import tqdm
    with torch.inference_mode():
        for start in tqdm(range(0, len(X), BATCH_SIZE), desc="RoBERTa"):
            batch = X.iloc[start:start + BATCH_SIZE].tolist()
            tok = tokenizer(batch, padding=True, truncation=True,
                            max_length=MAX_LENGTH, return_tensors="pt")
            tok = {k: v.to(device) for k, v in tok.items()}
            hidden = encoder(**tok).last_hidden_state
            mask = tok["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            arrays.append(pooled.cpu().float().numpy())
    embeddings = np.vstack(arrays).astype(np.float32)
    np.save(EMB_PATH, embeddings)
    del encoder
    if device.type == "mps":
        torch.mps.empty_cache()

assert embeddings.shape[0] == len(X)
assert np.isfinite(embeddings).all()
print("Embeddings:", embeddings.shape)

W1118 14:48:08.334000 48813 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19797.25it/s]
[transformers] RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
RoBERTa: 100%|██████████| 169/169 [00:29<00:00,  5.70it/s]

Embeddings: (2699, 768)


## 3. Explicit style features

Reuse the strongest groups from notebook 04: **structure + sentence starts**. Function-word features are excluded based on the earlier exploratory ablation. Text-by-text feature extraction does not use labels.

In [3]:
START_WORDS = [
    "beneath", "imagine", "at", "the", "a", "in", "as", "when", "if", "it",
    "you", "we", "there", "this", "but", "and", "for", "on", "with", "while",
]
WORD_RE = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
STRUCTURE = [
    "word_count", "sentence_count", "mean_word_len", "std_word_len",
    "mean_sent_len", "std_sent_len", "median_sent_len", "comma_per_100",
    "semicolon_per_100", "colon_per_100", "em_dash_per_100",
    "apostrophe_per_100", "question_per_100",
]
STARTS = [f"start_{w}" for w in START_WORDS] + ["start_other"]
STYLE_COLS = STRUCTURE + STARTS

def extract_style(text):
    tokens = [t.lower() for t in WORD_RE.findall(text)]
    lengths = np.array([len(t) for t in tokens], dtype=float)
    segments = [s.strip() for s in SENT_SPLIT_RE.split(text) if WORD_RE.search(s)]
    sentence_tokens = [[t.lower() for t in WORD_RE.findall(s)] for s in segments]
    sent_lengths = np.array([len(s) for s in sentence_tokens], dtype=float)
    first = [s[0] for s in sentence_tokens if s]
    starts = Counter(first)
    per100 = 100 / max(1, len(tokens))
    out = {
        "word_count": float(len(tokens)),
        "sentence_count": float(len(segments)),
        "mean_word_len": float(lengths.mean()) if len(lengths) else 0.,
        "std_word_len": float(lengths.std()) if len(lengths) else 0.,
        "mean_sent_len": float(sent_lengths.mean()) if len(sent_lengths) else 0.,
        "std_sent_len": float(sent_lengths.std()) if len(sent_lengths) else 0.,
        "median_sent_len": float(np.median(sent_lengths)) if len(sent_lengths) else 0.,
        "comma_per_100": text.count(",") * per100,
        "semicolon_per_100": text.count(";") * per100,
        "colon_per_100": text.count(":") * per100,
        "em_dash_per_100": text.count("—") * per100,
        "apostrophe_per_100": (text.count("'") + text.count("’")) * per100,
        "question_per_100": text.count("?") * per100,
    }
    out.update({f"start_{word}": starts[word] / max(1, len(first)) for word in START_WORDS})
    out["start_other"] = (1 - sum(out[f"start_{word}"] for word in START_WORDS)) if first else 0.
    return out

styles = pd.DataFrame.from_records([extract_style(text) for text in X], columns=STYLE_COLS)
assert np.isfinite(styles.to_numpy()).all()
print("Style features:", styles.shape)

Style features: (2699, 34)


## 4. Classification on held-out prompts

For each fold: fit TF-IDF, scalers and classifier **only on training texts**. We compare four feature sets. Fusion weights are fixed in advance; this is exploratory, not a nested-CV hyperparameter search.

In [4]:
# Same TF-IDF settings and 0.15 stylometry weight as notebook 04.
STYLE_WEIGHT = 0.15
EMB_WEIGHT = 1.0

variants = [
    "char", "char_style", "roberta", "roberta_style", "char_roberta", "all"
]
fold_rows = []
oof = {variant: np.empty(len(X), dtype=object) for variant in variants}

for fold_id, (tr, te) in enumerate(folds, start=1):
    vectorizer = TfidfVectorizer(
        analyzer="char", ngram_range=(2, 5), min_df=2,
        sublinear_tf=True, max_features=100_000,
    )
    char_tr = vectorizer.fit_transform(X.iloc[tr]); char_te = vectorizer.transform(X.iloc[te])

    # Scalers are fitted separately on the training portion of each fold.
    sty_scaler = StandardScaler()
    sty_tr = normalize(sty_scaler.fit_transform(styles.iloc[tr]).astype(np.float32))
    sty_te = normalize(sty_scaler.transform(styles.iloc[te]).astype(np.float32))
    sty_tr = sparse.csr_matrix(sty_tr); sty_te = sparse.csr_matrix(sty_te)

    emb_scaler = StandardScaler()
    emb_tr = normalize(emb_scaler.fit_transform(embeddings[tr]).astype(np.float32))
    emb_te = normalize(emb_scaler.transform(embeddings[te]).astype(np.float32))
    emb_tr = sparse.csr_matrix(emb_tr); emb_te = sparse.csr_matrix(emb_te)

    features = {
        "char": (char_tr, char_te),
        "char_style": (
            sparse.hstack([char_tr, STYLE_WEIGHT * sty_tr], format="csr"),
            sparse.hstack([char_te, STYLE_WEIGHT * sty_te], format="csr"),
        ),
        "roberta": (emb_tr, emb_te),
        "roberta_style": (
            sparse.hstack([EMB_WEIGHT * emb_tr, STYLE_WEIGHT * sty_tr], format="csr"),
            sparse.hstack([EMB_WEIGHT * emb_te, STYLE_WEIGHT * sty_te], format="csr"),
        ),
        "char_roberta": (
            sparse.hstack([char_tr, EMB_WEIGHT * emb_tr], format="csr"),
            sparse.hstack([char_te, EMB_WEIGHT * emb_te], format="csr"),
        ),
        "all": (
            sparse.hstack([char_tr, EMB_WEIGHT * emb_tr, STYLE_WEIGHT * sty_tr], format="csr"),
            sparse.hstack([char_te, EMB_WEIGHT * emb_te, STYLE_WEIGHT * sty_te], format="csr"),
        ),
    }

    for variant, (A_tr, A_te) in features.items():
        clf = LinearSVC(C=1.0, max_iter=5000)
        clf.fit(A_tr, y.iloc[tr])
        pred = clf.predict(A_te)
        oof[variant][te] = pred
        fold_rows.append({
            "fold": fold_id, "variant": variant,
            "balanced_accuracy": balanced_accuracy_score(y.iloc[te], pred),
        })
    print(f"Completed fold {fold_id}/5")

fold_results = pd.DataFrame(fold_rows)
summary = pd.DataFrame([
    {"variant": v,
     "balanced_accuracy_oof": balanced_accuracy_score(y, oof[v]),
     "accuracy_oof": accuracy_score(y, oof[v]),
     "mean_fold_bal_acc": fold_results.loc[fold_results.variant == v, "balanced_accuracy"].mean()}
    for v in variants
]).sort_values("balanced_accuracy_oof", ascending=False)
summary["delta_vs_char_style"] = summary["balanced_accuracy_oof"] - balanced_accuracy_score(y, oof["char_style"])
display(summary.round(4))
display(fold_results.pivot(index="fold", columns="variant", values="balanced_accuracy").round(4))

Completed fold 1/5
Completed fold 2/5
Completed fold 3/5
Completed fold 4/5
Completed fold 5/5


,variant,balanced_accuracy_oof,accuracy_oof,mean_fold_bal_acc,delta_vs_char_style
5,all,0.3823,0.3824,0.3823,0.0233
4,char_roberta,0.3756,0.3757,0.3756,0.0167
3,roberta_style,0.3623,0.3624,0.3623,0.0033
1,char_style,0.3589,0.3590,0.3590,0.0000
2,roberta,0.3475,0.3475,0.3475,-0.0115
0,char,0.3156,0.3157,0.3156,-0.0433


variant,all,char,char_roberta,char_style,roberta,roberta_style
fold,,,,,,
1,0.3852,0.3019,0.3852,0.3574,0.3407,0.3630
2,0.3556,0.2944,0.3481,0.3352,0.3370,0.3407
3,0.3796,0.3259,0.3833,0.3685,0.3704,0.3704
4,0.3833,0.3278,0.3630,0.3704,0.3278,0.3722
5,0.4077,0.3281,0.3985,0.3633,0.3615,0.3651


## 5. Diagnostics and conclusions

**Do not interpret a win on these folds as a final leaderboard improvement.** If you select a model or tune fusion weights based on these scores, validate the selected configuration using new prompts or nested GroupKFold.

In [5]:
# Show per-model recall changes against the char+style baseline.
from sklearn.metrics import recall_score
base_recall = recall_score(y, oof["char_style"], labels=LABELS, average=None, zero_division=0)
best_variant = summary.iloc[0]["variant"]
best_recall = recall_score(y, oof[best_variant], labels=LABELS, average=None, zero_division=0)
per_model = pd.DataFrame({
    "model": LABELS, "baseline_recall": base_recall,
    "selected_recall": best_recall,
    "delta": best_recall - base_recall,
}).sort_values("delta", ascending=False)
print("Best observed variant:", best_variant)
display(per_model.round(3))

Best observed variant: all


,model,baseline_recall,selected_recall,delta
16,kimi-k3,0.273,0.467,0.193
3,claude-opus-5,0.613,0.720,0.107
10,glm-5.3-flash,0.180,0.233,0.053
13,gpt-5.6-sol,0.160,0.213,0.053
1,claude-opus-4-6,0.507,0.553,0.047
0,claude-fable-5,0.107,0.148,0.040
15,gpt-6-astra,0.613,0.653,0.040
14,gpt-5.6-terra,0.273,0.313,0.040
2,claude-opus-4-8,0.267,0.300,0.033
12,gpt-5.6-luna,0.287,0.320,0.033


### Conclusions

- **Frozen RoBERTa** reached **34.75%** balanced accuracy, above character TF-IDF (**31.56%**).
- **Character TF-IDF + stylometry** reached **35.89%** on these same folds.
- **All features combined** (character TF-IDF, stylometry, frozen RoBERTa) achieved **38.23%** balanced accuracy (**+2.33 percentage points** versus char + stylometry).
- Gains differ across models: Kimi and Claude Opus 5 improved most, while Qwen and Gemini 3.6 Flash declined.

**Limitation:** These are model-selection results on five grouped folds, not an independent estimate of hidden-test performance. The next notebook evaluates **fine-tuning RoBERTa** on unseen prompts, with fixed training settings and no checkpoint selection using the test fold.

## 6. Save reproducible results

In [6]:
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
summary.to_csv(RESULTS_DIR / "05_roberta_summary.csv", index=False)
fold_results.to_csv(RESULTS_DIR / "05_roberta_folds.csv", index=False)
print("Saved:", RESULTS_DIR)

Saved: /Users/ulad/kaggle/name-that-llm/results
